In [1]:
# === PARAMETERS ===
MODEL_TYPE = "CNN" # "MLP" or "CNN"
LOG_FILE = "training_log.txt"

# Grid / Agents
WIDTH = 6
HEIGHT = 6
NUM_AGENTS = 2
SPAWN_PROB = 0.05
DESPAWN_PROB = 0.09

# MLP Configuration
MLP_HIDDEN_DIM = 512
MLP_NUM_LAYERS = 4

# CNN Configuration
CNN_CONV_CHANNELS = [16, 16]
CNN_HEAD_HIDDEN_DIM = 64
CNN_HEAD_NUM_LAYERS = 3
CNN_KERNEL_SIZE = 3

# Training
LR = 0.0001
BATCH_SIZE = 32
DISCOUNT = 0.0 # Immediate Reward (Regression)
STOP_THRESHOLD_MAE = 0.01 # 1% Error
SEED = 42
MAX_STEPS = 100000 # 500_000
EVAL_FREQ = 1000

In [2]:
import sys
import numpy as np
import torch
import random
from tqdm import tqdm
import matplotlib.pyplot as plt

# Add root to path (assuming notebook is in experiment folder)
sys.path.append("../") 

from tadd_helpers.env_functions import init_empty_state, spawn_apples, despawn_apples, State
from tadd_helpers.eval_functions import nearest_policy

from models.value_mlp import ValueMLPCentralized
from models.value_cnn_new import ValueCNNCentralized

# Initial Seed (Will be reset in loop)
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

--- PyTorch is configured to use: cuda ---


In [3]:
if MODEL_TYPE == "MLP":
    model = ValueMLPCentralized(
        height=HEIGHT, width=WIDTH, lr=LR, discount=DISCOUNT,
        hidden_dim=MLP_HIDDEN_DIM, num_layers=MLP_NUM_LAYERS
    )
elif MODEL_TYPE == "CNN":
    model = ValueCNNCentralized(
        height=HEIGHT, width=WIDTH, lr=LR, discount=DISCOUNT,
        hidden_dim=CNN_HEAD_HIDDEN_DIM, num_layers=CNN_HEAD_NUM_LAYERS, 
        conv_channels=CNN_CONV_CHANNELS, kernel_size=CNN_KERNEL_SIZE
    )
else:
    raise ValueError("Invalid MODEL_TYPE")

params = sum(p.numel() for p in model.policy_net.parameters())
print(f"Initialized {MODEL_TYPE}. Total Trainable Parameters: {params}")

Initialized CNN. Total Trainable Parameters: 47937


In [4]:
def generate_balanced_test_suite(n=500):
    states_zero = []
    states_pick = []
    
    # 1. Zeros (No agent on ANY apple)
    while len(states_zero) < n:
        s = init_empty_state(HEIGHT, WIDTH, NUM_AGENTS)
        # Add noise apples STRICTLY on empty tiles
        num_noise = np.random.randint(1, 6)
        cnt = 0; safety = 0
        while cnt < num_noise and safety < 100:
            r, c = np.random.randint(0, HEIGHT), np.random.randint(0, WIDTH)
            if s.agents[r, c] == 0 and s.apples[r, c] == 0:
                s.apples[r, c] = 1; cnt += 1
            safety += 1
        states_zero.append(s)
        
    # 2. Pickers (Exactly ONE agent on ONE apple)
    while len(states_pick) < n:
        s = init_empty_state(HEIGHT, WIDTH, NUM_AGENTS)
        s.apples[:] = 0
        
        # A. Pick Target
        picker_id = np.random.randint(0, NUM_AGENTS)
        r_t, c_t = np.random.randint(0, HEIGHT), np.random.randint(0, WIDTH)
        
        # B. Place Picker and Apple
        s.set_agent_position(picker_id, np.array([r_t, c_t]))
        s.apples[r_t, c_t] = 1
        
        # C. Ensure NO other agent is at (r_t, c_t)
        for other_id in range(NUM_AGENTS):
            if other_id != picker_id:
                while tuple(s.agent_position(other_id)) == (r_t, c_t):
                    r_new, c_new = np.random.randint(0, HEIGHT), np.random.randint(0, WIDTH)
                    s.set_agent_position(other_id, np.array([r_new, c_new]))

        # D. Add Noise Apples (Strictly on empty tiles)
        num_noise = np.random.randint(0, 5)
        cnt = 0; safety = 0
        while cnt < num_noise and safety < 100:
            r, c = np.random.randint(0, HEIGHT), np.random.randint(0, WIDTH)
            if s.agents[r, c] == 0 and s.apples[r, c] == 0:
                s.apples[r, c] = 1; cnt += 1
            safety += 1
            
        states_pick.append(s)
        
    return states_zero, states_pick

test_zeros, test_picks = generate_balanced_test_suite()
print(f"Test Suite: {len(test_zeros)} Zeros, {len(test_picks)} Pickers")

Test Suite: 500 Zeros, 500 Pickers


In [5]:
# =============================================================================
# 1. DEFINE ARCHITECTURE WITH GLOBAL MAX POOLING
# =============================================================================
import torch.nn as nn
import torch.nn.functional as F
from models.value_cnn_new import BaseValueModel
from config import DEVICE

class CNNGlobalMaxPool(nn.Module):
    def __init__(self, input_channels, height, width, conv_channels, hidden_dim, num_layers, kernel_size):
        super().__init__()
        self.conv_layers = nn.ModuleList()
        in_c = input_channels
        
        # 1. 1x1 Convolutions (Pixel-wise processing)
        for out_c in conv_channels:
            padding = (kernel_size - 1) // 2
            self.conv_layers.append(nn.Conv2d(in_c, out_c, kernel_size, padding=padding))
            self.conv_layers.append(nn.ReLU())
            in_c = out_c
            
        # 2. MLP Head (Now takes purely channel-dimension input)
        # Note: Input dim is now 'in_c', not 'in_c * H * W'
        layers = []
        in_dim = in_c 
        for _ in range(num_layers):
            layers.append(nn.Linear(in_dim, hidden_dim))
            layers.append(nn.ReLU())
            in_dim = hidden_dim
        layers.append(nn.Linear(in_dim, 1))
        
        self.mlp_head = nn.Sequential(*layers)

    def forward(self, x):
        # x shape: [Batch, Channels, Height, Width]
        for l in self.conv_layers:
            x = l(x)
        
        # --- GLOBAL MAX POOLING ---
        # "Is the feature present ANYWHERE in the grid?"
        # Output shape: [Batch, Channels, 1, 1]
        x = F.max_pool2d(x, kernel_size=x.size()[2:])
        
        # Flatten: [Batch, Channels]
        x = torch.flatten(x, 1)
        
        return self.mlp_head(x)

class ValueCNNCentralizedPool(BaseValueModel):
    def __init__(self, height, width, lr, discount, hidden_dim, num_layers, conv_channels, kernel_size):
        super().__init__(discount, 100000)
        # Use the New MaxPool Class
        self.policy_net = CNNGlobalMaxPool(2, height, width, conv_channels, hidden_dim, num_layers, kernel_size).to(DEVICE)
        self.target_net = CNNGlobalMaxPool(2, height, width, conv_channels, hidden_dim, num_layers, kernel_size).to(DEVICE)
        self.target_net.load_state_dict(self.policy_net.state_dict())
        self.target_net.eval()
        self.optimizer = torch.optim.AdamW(self.policy_net.parameters(), lr=lr, amsgrad=True)

    def raw_state_to_nn_input(self, state, **kwargs):
        return np.stack([state.apples, state.agents], axis=0).astype(np.float32)

# =============================================================================
# 2. INITIALIZE AND TRAIN
# =============================================================================
model = ValueCNNCentralizedPool(
    height=HEIGHT, width=WIDTH, lr=LR, discount=0.0,
    hidden_dim=CNN_HEAD_HIDDEN_DIM, num_layers=CNN_HEAD_NUM_LAYERS, 
    conv_channels=CNN_CONV_CHANNELS, kernel_size=CNN_KERNEL_SIZE
)

print(f"Initialized CNN (Global Max Pool). Params: {sum(p.numel() for p in model.policy_net.parameters())}")


Initialized CNN (Global Max Pool). Params: 12097


In [6]:
# --- TRAINING LOOP ---
loss_history = []
max_ape_history = []

torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

# Reset Buffer
model.memory.memory.clear()

s: State = init_empty_state(HEIGHT, WIDTH, NUM_AGENTS)
LOG_FILE = "training_log.txt"
BATCH_SIZE = 64

with open(LOG_FILE, "a") as f:
    f.write(f"\n=== STARTING CENTRALIZED TRAINING (CNN-MAX-POOL) [Reward=1.0] ===\n")
    f.write("Step | Loss    | Max%_Z | Avg%_Z | Max%_P | Avg%_P\n")

pbar = tqdm(range(MAX_STEPS))
for step in pbar:
    
    # 1. MOVE (Reward 0)
    active_agent_idx = s.get_random_agent_id()
    move_action = nearest_policy(s, active_agent_idx)
    new_pos = np.clip(s.agent_position(active_agent_idx) + move_action.vector, [0, 0], [HEIGHT-1, WIDTH-1])
    s_mid = s.copy()
    s_mid.set_agent_position(active_agent_idx, new_pos)
    
    model.add_experience(s, s_mid, 0.0)

    # 2. PICK (Reward 1.0 or 0.0)
    s_next = s_mid.copy()
    reward = 0.0
    if s_next.apples[tuple(new_pos)] > 0:
        s_next.remove_apple_at(new_pos)
        reward = 1.0 # <--- CHANGED FROM -1.0 TO 1.0
        
    spawn_apples(s_next, SPAWN_PROB)
    despawn_apples(s_next, DESPAWN_PROB)
    
    if reward != 0:
        model.add_experience(s_mid, s_next, reward)
    elif random.random() < 0.5:
        model.add_experience(s_mid, s_next, 0.0)

    # 3. TRAIN
    loss = model.train_batch(BATCH_SIZE)
    if loss is not None: loss_history.append(loss)
    s = s_next
    
    # 4. EVAL
    if step % EVAL_FREQ == 0 and step > 0:
        preds_z = np.array([model.get_value(st) for st in test_zeros])
        if len(preds_z) > 0:
            max_ape_z = np.max(np.abs(preds_z)) * 100
            avg_ape_z = np.mean(np.abs(preds_z)) * 100
        else: max_ape_z, avg_ape_z = 0, 0
        
        preds_p = np.array([model.get_value(st) for st in test_picks])
        if len(preds_p) > 0:
            # <--- CHANGED TARGET TO 1.0
            max_ape_p = np.max(np.abs(preds_p - 1.0)) * 100
            avg_ape_p = np.mean(np.abs(preds_p - 1.0)) * 100
        else: max_ape_p, avg_ape_p = 0, 0
        
        global_max = max(max_ape_z, max_ape_p)

        def log_buffer_stats(model):
            r = [t.reward for t in model.memory.memory]
            if not r: return 0,0
            z = r.count(0.0)
            return (z/len(r))*100, ((len(r)-z)/len(r))*100
        
        pct_z, pct_ev = log_buffer_stats(model)

        with open(LOG_FILE, "a") as f:
            f.write(f"{step:<6} | {loss:.5f} | {max_ape_z:6.2f} | {avg_ape_z:6.2f} | {max_ape_p:6.2f} | {avg_ape_p:6.2f} | BZ:{pct_z:.0f}%\n")
            
        pbar.set_description(f"L:{loss:.4f} | Max%:{global_max:.0f}%")
        
        if global_max < 1.0:
            print(f"✅ CONVERGED at Step {step}")
            break

  0%|          | 0/100000 [00:00<?, ?it/s]

L:0.0000 | Max%:27%:  43%|████▎     | 42835/100000 [03:24<04:33, 209.24it/s]


KeyboardInterrupt: 